## Setup and imports

In [ ]:
EXP_NAME = "10MAY_simple_synth_unbiased_2"
dataset = "synth/9MAY/simple_unbiased_train"
feature_map = "sps/features_synth_simple_unbiased_ind"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.feature_selection import mutual_info_regression
from sklearn.utils import resample

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{dataset}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

In [ ]:
# sps_audit.head()

# Coverage

In [ ]:
iteration_per_feature = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('feature')[['iteration']].count()
iteration_per_feature

In [ ]:
group_mace_cols = sps_audit.columns.str.startswith('mace_')
group_mace_baseline_cols = sbs_audit_baseline.columns.str.startswith('mace_')
sps_audit['max_mace'] = sps_audit.loc[:, group_mace_cols].max(axis=1)
sbs_audit_baseline['max_mace'] = sbs_audit_baseline.loc[:, group_mace_baseline_cols].max(axis=1)

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)

feature_desc_stats = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['feature'])[['te_error', 'max_mace', 'auprc']].median()
feature_corr_stats = full_audit[full_audit['bucket'] == 'x_corr'].groupby(['feature'])[['te_error', 'max_mace', 'auprc']].median()

# Config-level trade-off analyses

In [ ]:
agg_sbs_audit_baseline = sbs_audit_baseline.groupby('iteration')

baseline_te_error = agg_sbs_audit_baseline['te_error'].agg('median').values[0]
baseline_mace = agg_sbs_audit_baseline['mace'].agg('median').values[0]
baseline_max_mace = agg_sbs_audit_baseline['max_mace'].agg('median').values[0]
baseline_roc_auc = agg_sbs_audit_baseline['roc_auc'].agg('median').values[0]
baseline_auprc = agg_sbs_audit_baseline['auprc'].agg('median').values[0]

In [ ]:
tradeoff_audit = full_audit.groupby('iteration')[['te_error', 'max_mace', 'auprc']].median().round(3)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['iteration'])['feature'].apply(set).to_dict()

## Utility vs Fairness trade-off

In [ ]:
print(f"Baseline TE error: {baseline_te_error:.3f}")
print(f"Baseline Average Precision: {baseline_auprc:.3f}")
print(f"Baseline ROC AUC: {baseline_roc_auc:.3f}")
print(f"Baseline MACE: {baseline_mace:.3f}")
print(f"Baseline max MACE: {baseline_max_mace:.3f}")

In [ ]:
utility_vs_fairness = tradeoff_audit.sort_values('auprc', ascending=False)

positives = dataset[dataset[features["target"]["name"]] == 1]
auprc_baseline = len(positives) / len(dataset)
print(f'Chance-level AUPRC: {auprc_baseline:.3f}')

pareto_frontier = []
current_min_mace = 1

print('\n--- Configurations on the AUPRC / max group MACE Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['max_mace'] < current_min_mace:
    if len(pareto_frontier) > 0:
      solution['delta_mace'] = (solution['max_mace'] - pareto_frontier[0]['max_mace']) / pareto_frontier[0]['max_mace']
      solution['delta_auprc'] = (solution['auprc'] - pareto_frontier[0]['auprc']) / (pareto_frontier[0]['auprc'] - auprc_baseline)
      solution['FEU'] = solution['delta_mace'] / solution['delta_auprc']
    pareto_frontier.append(solution)
    current_min_mace = solution['max_mace']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(baseline_auprc, baseline_max_mace, marker="D", color="#D92B89", linestyle="")
sns.lineplot(data=pareto_frontier_df, x='auprc', y='max_mace', color="green", marker='', linestyle="--", errorbar=None, ax=ax)
sns.scatterplot(data=utility_vs_fairness, x='auprc', y='max_mace', ax=ax)
ax.axvline(auprc_baseline)
plt.xlabel('Average Precision (AUPRC)')
plt.ylabel('Max group MACE')
plt.ylim(bottom=0)
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit',
                   'Chance-level AUPRC'
                   ],
           loc='upper left', bbox_to_anchor=(0, -0.1), edgecolor="white")

plt.show()

# All Configs explored

In [ ]:
def find_config(row):
  return x_desc_configs.get(row['id'], "empty Xdesc")

all_configs = utility_vs_fairness.sort_values('max_mace').reset_index(names="id")
all_configs['Xdesc config'] = all_configs.apply(find_config, axis=1)

lowest_te_error = all_configs.sort_values('te_error').reset_index().loc[0, :]
lowest_max_mace = all_configs.sort_values('max_mace').reset_index().loc[0, :]
highest_auprc  = all_configs.sort_values('auprc', ascending=False).reset_index().loc[0, :]

print(f'Config with the lowest TE error: {lowest_te_error['Xdesc config']}\n')
print(f'Config with the lowest max MACE: {lowest_max_mace['Xdesc config']}\n')
print(f'Config with the highest AUPRC: {highest_auprc['Xdesc config']}\n')
print(all_configs.to_markdown())